In [28]:
import pandas as pd
import numpy as np
import os


# ===== FILE PATHS =====
MOTIF_FILE = "esm1b/conserved_hits_updated.xlsx"
LLR_DIRECTORY = "esm1b"
OUTPUT_FILE = "all_residues_fixeddecimalpoint/esm1b.csv"
# ===============================

# ===== CONSERVATION THRESHOLDS ===== 
STRONG_CONSERVATION_THRESHOLD = -7.5    # All non-WT amino acids must be below this
MODERATE_CONSERVATION_THRESHOLD = -7  # For a secondary flag
MIN_DELETERIOUS_FRACTION = 0.9          # What fraction of non-WT AAs must be below threshold
# ===============================

IDR_POS = {
    "Lsr2": [(61, 76)],
    "EspD": [(1, 14), (35, 60), (171, 184)],
    "EspA": [(270, 294), (301, 383)],
    "EspM": [(1, 34), (131, 161), (230, 249)],
    "EspE": [(204, 277), (361, 402)],
    "EspH": [(1, 33)],
    "PPE68": [(173, 202), (236, 368)],
    "EspI": [(1, 246), (257, 311), (325, 373)],
    "EspJ": [(97, 113), (176, 280)],
    "EspK": [(175, 461)],
    "EspB": [(93, 124), (265, 335), (352, 433), (453, 460)]
}


In [23]:
def read_llr_csv_with_locale_fix(filepath):
    """
    Read LLR CSV file and fix locale-formatted numbers (string numbers)
    """
    try:
        # First, try reading normally
        df = pd.read_csv(filepath, index_col=0)
        # Check if the amino acid columns contain strings (indicating locale formatting)
        amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
        # Look for the first amino acid column that exists
        test_col = None
        for aa in amino_acids:
            if aa in df.columns:
                test_col = aa
                break
        if test_col is None:
            print("No amino acid columns found")
            return None
        # Check if values are strings (indicating locale formatting issue)
        sample_value = df[test_col].iloc[0]    
        if isinstance(sample_value, str) and '.' in sample_value:
            print("Detected locale-formatted numbers, fixing...")        
            # Fix each amino acid column
            for aa in amino_acids:
                if aa in df.columns:
                    df[aa] = df[aa].apply(fix_locale_number)    
        return df
    except Exception as e:
        print(f" Error reading CSV: {e}")
        return None

def fix_locale_number(value):
    """
    Convert locale-formatted number string to float
    Examples:
    '-8.772.611.091.728.320' -> -8.772611091728320
    '0.0' -> 0.0
    """
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float)):
        return float(value)
    str_val = str(value).strip()
    # Handle simple cases
    if str_val == '0.0' or str_val == '0':
        return 0.0
    # For negative numbers, handle the sign separately
    is_negative = str_val.startswith('-')
    if is_negative:
        str_val = str_val[1:]
    # Split by periods
    parts = str_val.split('.')
    if len(parts) == 1:
        # No periods, just a simple number
        result = float(str_val)
    elif len(parts) == 2:
        # Could be either decimal or thousands separator
        # Check length of last part to guess
        if len(parts[1]) <= 3:
            # Likely a decimal (e.g., "123.45")
            result = float(str_val)
        else:
            # Likely thousands separator (e.g., "1.234567")
            result = float(''.join(parts))
    else:
        # Multiple periods - definitely thousands separators
        # Join all parts together
        result = float(''.join(parts))
    return -result if is_negative else result

def analyze_conservation(result, wt_aa):
    """
    Analyze conservation at a position based on LLR values
    Returns conservation metrics
    """
    amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
    # Get LLR values for non-wild-type amino acids
    non_wt_llrs = []
    for aa in amino_acids:
        if aa != wt_aa:  # Skip wild-type
            llr_val = result.get(f'LLR_{aa}')
            if not pd.isna(llr_val):
                non_wt_llrs.append(llr_val)
    if len(non_wt_llrs) == 0:
        return {
            'strongly_conserved': False,
            'moderately_conserved': False,
            'fraction_deleterious_strong': np.nan,
            'fraction_deleterious_moderate': np.nan,
            'max_non_wt_llr': np.nan,
            'mean_non_wt_llr': np.nan,
            'min_non_wt_llr': np.nan
        }
    non_wt_llrs = np.array(non_wt_llrs)
    max_non_wt_llr = np.max(non_wt_llrs)
    min_non_wt_llr = np.min(non_wt_llrs)
    mean_non_wt_llr = np.mean(non_wt_llrs)
    strongly_deleterious = np.sum(non_wt_llrs < STRONG_CONSERVATION_THRESHOLD)
    moderately_deleterious = np.sum(non_wt_llrs < MODERATE_CONSERVATION_THRESHOLD)
    fraction_strongly_del = strongly_deleterious / len(non_wt_llrs)
    fraction_moderately_del = moderately_deleterious / len(non_wt_llrs)
    # Conservation flags
    strongly_conserved = (
        fraction_strongly_del >= MIN_DELETERIOUS_FRACTION and
        max_non_wt_llr < STRONG_CONSERVATION_THRESHOLD
    )
    moderately_conserved = (
        fraction_moderately_del >= MIN_DELETERIOUS_FRACTION and
        max_non_wt_llr < MODERATE_CONSERVATION_THRESHOLD
    )
    return {
        'conserved': strongly_conserved,
        'borderline_conserved': moderately_conserved,
        'fraction_deleterious_strong': fraction_strongly_del,
        'fraction_deleterious_moderate': fraction_moderately_del,
        'max_non_wt_llr': max_non_wt_llr,
        'mean_non_wt_llr': mean_non_wt_llr,
        'min_non_wt_llr': min_non_wt_llr
    }

In [24]:
def analyze_full_proteins(output_f, motif_f, directory_f, disorder_dict):
    """
    Analyze conservation at EVERY residue of EVERY protein with an LLR file.
    Adds a flag indicating whether each residue is part of a motif/conserved IDR.
    """
    print("=== Full-Protein LLR Conservation Analysis ===\n")
    motif_positions = {}
    if os.path.exists(motif_f):
        if motif_f.endswith(('.xlsx', '.xls')):
            motif_df = pd.read_excel(motif_f)
        else:
            motif_df = pd.read_csv(motif_f)
        for _, row in motif_df.iterrows():
            protein_raw = row.get('Protein', '')
            if pd.isna(protein_raw):
                continue
            protein = str(protein_raw).lower().strip()
            if not protein:
                continue
            try:
                start = int(row['Motif_Start'])
                end = int(row['Motif_End'])
            except (ValueError, TypeError):
                continue
            motif_positions.setdefault(protein, set()).update(range(start, end + 1))
        print(f"Loaded motif data: {len(motif_df)} motif rows across {len(motif_positions)} proteins\n")
    else:
        print(f"No motif file found at {motif_f} — proceeding without motif annotation\n")

    disorder_positions = {}
    if disorder_dict:
        for protein, ranges in disorder_dict.items():
            protein_key = protein.lower().strip()
            disorder_positions[protein_key] = set()
            for start, end in ranges:
                disorder_positions[protein_key].update(range(start, end + 1))
        print(f"Loaded disorder data for {len(disorder_positions)} proteins\n")
    else:
        print("No disorder dictionary provided — proceeding without disorder annotation\n")

    if not os.path.exists(directory_f):
        print(f"LLR directory not found: {directory_f}")
        return
    all_llr_files = [f for f in os.listdir(directory_f) 
                     if f.lower().endswith('_llr_heatmap.csv')]
    print(f"Found {len(all_llr_files)} LLR files\n")
    results = []

    for filename in all_llr_files:
        protein = filename.replace('_LLR_heatmap.csv', '').replace('_llr_heatmap.csv', '')
        protein_key = protein.lower().strip()
        filepath = os.path.join(directory_f, filename)    
        print(f"Processing {protein}...")
        llr_df = read_llr_csv_with_locale_fix(filepath)
        if llr_df is None:
            continue    
        protein_motif_positions = motif_positions.get(protein_key, set())
        protein_disorder_positions = disorder_positions.get(protein_key, set())

        for _, pos_data in llr_df.iterrows():
            pos = int(pos_data['position'])
            wt_aa = pos_data['WT_residue']
            result = {
                'Protein': protein,
                'Position': pos,
                'WT_AA': wt_aa,
                'In_Motif': pos in protein_motif_positions,
                'In_IDR': pos in protein_disorder_positions, 
            }
            for aa in 'ACDEFGHIKLMNPQRSTVWY':
                if aa in llr_df.columns:
                    result[f'LLR_{aa}'] = pos_data[aa]
                else:
                    result[f'LLR_{aa}'] = np.nan        
            result['LLR_WT'] = result.get(f'LLR_{wt_aa}', np.nan)        
            # Conservation analysis
            conservation_metrics = analyze_conservation(result, wt_aa)
            result.update(conservation_metrics)
            results.append(result)    
        n_motif_in_protein = len(protein_motif_positions)
        n_disorder_in_protein = len(protein_disorder_positions)
        print(f"  → {len(llr_df)} residues analyzed ({n_motif_in_protein} in motifs), {n_disorder_in_protein} in disorder)")
    if not results:
        print("\n No results extracted")
        return
    # Save and summarize
    results_df = pd.DataFrame(results)
    results_df.to_csv(output_f, index=False, float_format='%.2f')  
    return results_df

In [29]:
analyze_full_proteins(OUTPUT_FILE, MOTIF_FILE, LLR_DIRECTORY, IDR_POS)

=== Full-Protein LLR Conservation Analysis ===

Loaded motif data: 23 motif rows across 8 proteins

Loaded disorder data for 11 proteins

Found 11 LLR files

Processing EspA...
  → 392 residues analyzed (6 in motifs), 108 in disorder)
Processing EspB...
  → 460 residues analyzed (8 in motifs), 193 in disorder)
Processing EspD...
  → 184 residues analyzed (0 in motifs), 54 in disorder)
Processing EspE...
  → 402 residues analyzed (3 in motifs), 116 in disorder)
Processing EspH...
  → 183 residues analyzed (0 in motifs), 33 in disorder)
Processing EspI...
  → 666 residues analyzed (19 in motifs), 350 in disorder)
Processing EspJ...
  → 280 residues analyzed (9 in motifs), 122 in disorder)
Processing EspK...
  → 729 residues analyzed (6 in motifs), 287 in disorder)
Processing EspM...
  → 392 residues analyzed (10 in motifs), 85 in disorder)
Processing Lsr2...
  → 112 residues analyzed (0 in motifs), 16 in disorder)
Processing PPE68...
  → 368 residues analyzed (6 in motifs), 163 in disord

,Protein,Position,WT_AA,In_Motif,In_IDR,LLR_A,LLR_C,LLR_D,LLR_E,LLR_F,...,LLR_W,LLR_Y,LLR_WT,conserved,borderline_conserved,fraction_deleterious_strong,fraction_deleterious_moderate,max_non_wt_llr,mean_non_wt_llr,min_non_wt_llr
0,EspA,1,M,False,False,-9.946927,-12.168204,-10.736062,-10.799422,-10.886605,...,-12.277226,-11.555199,0.0,True,True,1.0,1.000000,-8.945540,-10.686969,-12.277226
1,EspA,2,S,False,False,-0.803761,-4.522482,-2.921166,-3.500688,-3.183759,...,-4.454970,-4.604922,0.0,False,False,0.0,0.000000,-0.724645,-3.020994,-5.025827
2,EspA,3,R,False,False,1.611374,-2.274941,1.360240,1.213855,1.412952,...,-0.130609,0.072423,0.0,False,False,0.0,0.000000,2.034456,0.674256,-2.274941
3,EspA,4,A,False,False,0.000000,-2.923645,-0.060710,-0.217910,-0.232977,...,-1.864848,-0.950200,0.0,False,False,0.0,0.000000,1.306818,-0.598639,-2.923645
4,EspA,5,F,False,False,0.938684,-3.087125,0.547616,0.565768,0.000000,...,-0.737674,-0.881624,0.0,False,False,0.0,0.000000,1.404427,-0.062769,-3.087125
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4163,PPE68,364,E,False,True,-2.118241,-6.007288,0.605534,0.000000,-2.183589,...,-1.217942,-2.380138,0.0,False,False,0.0,0.000000,0.605534,-2.493487,-6.007288
4164,PPE68,365,E,False,True,-2.807274,-7.133680,0.428248,0.000000,-3.210992,...,-2.429685,-3.149514,0.0,False,False,0.0,0.052632,0.428248,-3.493881,-7.133680
4165,PPE68,366,D,False,True,-2.822962,-6.300466,0.000000,-0.113436,-1.803050,...,-2.003949,-2.717226,0.0,False,False,0.0,0.000000,-0.113436,-2.965284,-6.300466
4166,PPE68,367,D,False,True,-2.884185,-6.635387,0.000000,-0.177461,-3.050661,...,-1.644637,-2.990556,0.0,False,False,0.0,0.000000,-0.177461,-3.106605,-6.635387
